# 04 — Qualitative analysis and source-attribution audit

**Input:** local, restricted Taguette SQLite database. **Output:** final-theme aggregate summaries and private quotation/source-attribution audits.

Requires the completed manually coded Taguette project. It preserves the researcher coding and thematic decisions, and verifies quotation identifiers against original responses. **Do not publish the database, quotation exports, or participant-level outputs.**

In [ ]:
# Import libraries and define project paths

import re
import sqlite3
from html import unescape
from pathlib import Path

import pandas as pd

# Locate the project root regardless of the notebook folder.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if (
            folder
            / "data"
            / "processed"
            / "qualitative"
            / "taguette"
            / "2026-08-19_LLM Anxiety Qualitative Analysis.sqlite3"
        ).is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Cannot locate the Taguette database from {cwd}. "
        "Check that the database exists in the project's data directory."
    )

# Define qualitative analysis directories.
QUAL_DIR = PROJECT_ROOT / "data" / "processed" / "qualitative"
TAGUETTE_DIR = QUAL_DIR / "taguette"
EXPORT_DIR = QUAL_DIR / "exports"

print("Project root:", PROJECT_ROOT)
print("Qualitative data folder:", QUAL_DIR)
print("Export folder:", EXPORT_DIR)

In [ ]:
# Locate the completed taguette database

TAGUETTE_DB = (
    TAGUETTE_DIR
    / "2026-08-19_LLM Anxiety Qualitative Analysis.sqlite3"
)

if not TAGUETTE_DB.exists():
    raise FileNotFoundError(
        f"Taguette database not found:\n{TAGUETTE_DB}\n\n"
        "Check the filename in TAGUETTE_DB."
    )

print("Taguette database found:")
print(TAGUETTE_DB)

In [ ]:
# Open taguette database in read-only mode

# The Taguette project is our completed qualitative coding source.
# We open the SQLite database in read-only mode so this notebook
# can inspect and analyse the coding without modifying the
# original Taguette project.

db_uri = f"file:{TAGUETTE_DB}?mode=ro"

conn = sqlite3.connect(
    db_uri,
    uri=True
)

print("Database connection opened successfully in read-only mode.")

In [ ]:
# Validate the completed taguette project
# Before extracting any qualitative data, we verify the basic
# contents of the Taguette database.
#
# This is a sanity check only. These counts are not thematic
# findings; they help confirm that the correct completed project
# has been opened.

project_summary = pd.read_sql_query(
    """
    SELECT
        (SELECT COUNT(*) FROM documents) AS documents,
        (SELECT COUNT(*) FROM tags) AS tags,
        (SELECT COUNT(*) FROM highlights) AS highlights,
        (SELECT COUNT(*) FROM highlight_tags) AS tag_assignments
    """,
    conn
)

print("Completed Taguette project summary:")
display(project_summary)

In [ ]:
# Extract the final codebook
# The codebook is read directly from the completed Taguette
# project. This preserves the final code names and descriptions
# used during manual coding.
#
# We also calculate how many coded highlights were assigned to
# each code. These frequencies are descriptive summaries of the
# coding, not measures of thematic importance.

codebook = pd.read_sql_query(
    """
    SELECT
        t.id AS tag_id,
        t.path AS code,
        t.description AS description,
        COUNT(ht.highlight_id) AS n_highlights
    FROM tags AS t
    LEFT JOIN highlight_tags AS ht
        ON t.id = ht.tag_id
    GROUP BY
        t.id,
        t.path,
        t.description
    ORDER BY
        n_highlights DESC,
        code ASC
    """,
    conn
)

print(f"Codes in final Taguette codebook: {len(codebook)}")
display(codebook)

In [ ]:
# Create the active analytical codebook
# The Taguette database still contains one unused code:
# REVIEW_BURDEN.
#
# Because it was not applied to any excerpt, it does not form
# part of the final coded dataset. We preserve it in the raw
# extracted codebook for transparency, but exclude unused codes
# from the analytical codebook used in subsequent summaries.

active_codebook = (
    codebook[
        codebook["n_highlights"] > 0
    ]
    .reset_index(drop=True)
)

print(
    "Codes stored in Taguette:",
    len(codebook)
)

print(
    "Codes used in final coding:",
    len(active_codebook)
)

display(active_codebook)

In [ ]:
# Extract and clean all coded highlights
# Each row represents one code assignment to one highlighted
# excerpt from the completed Taguette project.
#
# A single highlight may carry more than one code, so the number
# of rows can be larger than the number of unique highlights.
#
# Taguette stores highlight snippets with HTML markup such as
# <p>...</p>. We preserve the original raw snippet and create a
# cleaned text version for reading, quotation selection, and
# exported analysis tables.

coded_highlights = pd.read_sql_query(
    """
    SELECT
        h.id AS highlight_id,
        h.document_id,
        h.start_offset,
        h.end_offset,
        h.snippet AS highlight_text,
        d.name AS document,
        t.path AS code
    FROM highlights AS h
    JOIN documents AS d
        ON h.document_id = d.id
    JOIN highlight_tags AS ht
        ON h.id = ht.highlight_id
    JOIN tags AS t
        ON ht.tag_id = t.id
    ORDER BY
        h.id,
        t.path
    """,
    conn
)


def clean_highlight_text(text):
    if pd.isna(text):
        return text

    text = unescape(str(text))
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


coded_highlights["highlight_clean"] = (
    coded_highlights["highlight_text"]
    .apply(clean_highlight_text)
)


print(
    "Unique coded highlights:",
    coded_highlights["highlight_id"].nunique()
)

print(
    "Highlight-code assignments:",
    len(coded_highlights)
)

display(
    coded_highlights[
        [
            "highlight_id",
            "document",
            "code",
            "highlight_clean",
        ]
    ].head(15)
)

In [ ]:
# Recover source identifiers from complete taguette headers
# The original offset-based search occasionally stopped *inside* a
# response header and captured truncated IDs (P0, P01, P07...).
# We search the complete document first, then associate each highlight
# with the full header containing/preceding its start offset.
# IDs are source identifiers, not automatically sequential survey IDs.

import bisect

document_contents = pd.read_sql_query(
    "SELECT id AS document_id, contents FROM documents", conn
)
contents_by_document = dict(zip(
    document_contents["document_id"], document_contents["contents"]
))

participant_header_pattern = re.compile(
    r"Response\s+(R\d{3})\s*\|\s*Participant\s+(P\d{3})(?!\d)"
)

headers_by_document = {}
for document_id, contents in contents_by_document.items():
    matches = list(participant_header_pattern.finditer(contents))
    if not matches:
        raise ValueError(f"No complete source response headers in document {document_id}")
    headers_by_document[document_id] = [
        (m.start(), m.end(), m.group(1), m.group(2)) for m in matches
    ]

highlight_metadata = coded_highlights[
    ["highlight_id", "document_id", "start_offset", "end_offset"]
].drop_duplicates("highlight_id").copy()

# When a highlight begins inside a header, assign the *complete* header;
# otherwise use the most recent header at or before its start.
def source_ids_for_highlight(row):
    headers = headers_by_document[row["document_id"]]
    starts = [h[0] for h in headers]
    position = bisect.bisect_right(starts, int(row["start_offset"])) - 1
    if position < 0:
        raise ValueError(f"Highlight {row['highlight_id']} occurs before any source header")
    header_start, header_end, response_id, participant_id = headers[position]
    overlaps_header = int(row["start_offset"]) < header_end
    return pd.Series({
        "response_id": response_id,
        "participant_id": participant_id,
        "header_overlap": overlaps_header,
    })

recovered_metadata = highlight_metadata.apply(source_ids_for_highlight, axis=1)
highlight_metadata = pd.concat(
    [highlight_metadata[["highlight_id"]], recovered_metadata], axis=1
)
coded_highlights = coded_highlights.merge(
    highlight_metadata, on="highlight_id", how="left", validate="many_to_one"
)

# Explicit provenance checks before any thematic or quotation analysis.
assert coded_highlights[["participant_id", "response_id"]].notna().all().all()
assert coded_highlights["participant_id"].str.fullmatch(r"P\d{3}").all()
assert coded_highlights["response_id"].str.fullmatch(r"R\d{3}").all()
assert coded_highlights.groupby("highlight_id")["participant_id"].nunique().max() == 1
assert coded_highlights.groupby("highlight_id")["response_id"].nunique().max() == 1

source_pairs = {
    (h[2], h[3])
    for headers in headers_by_document.values()
    for h in headers
}
coded_pairs = set(map(tuple, coded_highlights[["response_id", "participant_id"]]
                      .drop_duplicates().itertuples(index=False, name=None)))
assert coded_pairs <= source_pairs, "Recovered identifiers absent from source headers"

print("Complete source headers:", sum(map(len, headers_by_document.values())))
print("Highlights overlapping source headers:",
      int(highlight_metadata["header_overlap"].sum()))
print("Unique coded participants:", coded_highlights["participant_id"].nunique())
print("Unique coded responses:", coded_highlights["response_id"].nunique())

# Source-level structural checks: these values are independently
# described in the completed thesis, not tuned statistical targets.
assert len(source_pairs) == 429, "Expected 429 source response pairs; inspect source database"
assert len({p for _, p in source_pairs}) == 118, "Expected 118 source respondents"
assert coded_highlights["participant_id"].nunique() <= 118


In [ ]:
# Check completeness of source-derived response identifiers.
# Historical case-specific mappings are retained only in the private research notebook.
required_columns = {"highlight_id", "participant_id", "response_id"}
assert required_columns.issubset(highlight_metadata.columns)
assert highlight_metadata[["participant_id", "response_id"]].notna().all().all()
print("PASS: source-derived participant and response identifiers are present.")

In [ ]:
# Summarise code frequencies
# This step provides two descriptive measures for each code:
#
#   1. n_highlights:
#      how many distinct coded excerpts were assigned the code
#
#   2. n_participants:
#      how many unique participants contributed at least one
#      excerpt carrying the code
#
# These frequencies help describe the coded dataset, but they
# do not determine thematic importance.

code_frequencies = (
    coded_highlights
    .groupby("code")
    .agg(
        n_highlights=(
            "highlight_id",
            "nunique"
        ),
        n_participants=(
            "participant_id",
            "nunique"
        )
    )
    .reset_index()
    .merge(
        active_codebook[
            [
                "code",
                "description",
            ]
        ],
        on="code",
        how="left"
    )
    [
        [
            "code",
            "description",
            "n_highlights",
            "n_participants",
        ]
    ]
    .sort_values(
        [
            "n_highlights",
            "n_participants",
            "code",
        ],
        ascending=[
            False,
            False,
            True,
        ]
    )
    .reset_index(drop=True)
)

print(
    "Active codes summarised:",
    len(code_frequencies)
)

display(
    code_frequencies.head(25)
)

In [ ]:
# Summarise code co-occurrences
# A co-occurrence means that two codes were assigned to the
# same highlighted excerpt.
#
# This can help reveal relationships between codes, but it does
# not automatically define themes. Themes remain interpretive
# patterns developed through review of the coded material.

highlight_code_matrix = (
    coded_highlights
    .assign(value=1)
    .pivot_table(
        index="highlight_id",
        columns="code",
        values="value",
        aggfunc="max",
        fill_value=0
    )
)

# Count how often each pair of codes occurs on the same highlight.
cooccurrence_matrix = (
    highlight_code_matrix.T
    @ highlight_code_matrix
)

# Set the diagonal to zero.
# The diagonal represents a code co-occurring with itself,
# which is not meaningful for this analysis.
#
# We use .loc instead of np.fill_diagonal() because some recent
# pandas/NumPy versions expose the underlying array as read-only.

for code_name in cooccurrence_matrix.index:
    cooccurrence_matrix.loc[
        code_name,
        code_name
    ] = 0


# Convert the matrix into a simpler table of unique code pairs.
cooccurrence_pairs = []

codes = list(
    cooccurrence_matrix.columns
)

for i, code_1 in enumerate(codes):

    for code_2 in codes[i + 1:]:

        count = int(
            cooccurrence_matrix.loc[
                code_1,
                code_2
            ]
        )

        if count > 0:

            cooccurrence_pairs.append(
                {
                    "code_1": code_1,
                    "code_2": code_2,
                    "n_cooccurrences": count,
                }
            )


cooccurrence_pairs = (
    pd.DataFrame(
        cooccurrence_pairs
    )
    .sort_values(
        "n_cooccurrences",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "Code pairs with at least one co-occurrence:",
    len(cooccurrence_pairs)
)

display(
    cooccurrence_pairs.head(25)
)

In [ ]:
# Group related codes into provisional analytic clusters
# The first interpretive step is not to name themes immediately.
# Instead, related codes are grouped into broader analytic
# clusters based on conceptual similarity and relationships
# observed during coding.
#
# These clusters are provisional. They are used to retrieve and
# review the underlying participant excerpts before deciding
# whether a coherent theme exists.

analytic_clusters = {
    "Support and manageability": [
        "STRESS_REDUCTION",
        "TASK_COMPLETION_SAFETY_NET",
        "TIME_PRESSURE_RELIEF",
        "BREAKING_TASK_PARALYSIS",
        "ALWAYS_AVAILABLE_SUPPORT",
        "REASSURANCE",
        "TASK_CLARIFICATION",
    ],

    "Learning access and explanation": [
        "LEARNING_SUPPORT",
        "PERSONALIZED_EXPLANATION",
        "AI_AS_TUTOR",
        "LOW_BARRIER_QUESTIONING",
        "TASK_STRUCTURING",
        "LEARNING_CATCH_UP",
        "CODE_EXPLANATION",
        "ERROR_EXPLANATION",
        "KNOWLEDGE_CHECKING",
    ],

    "Confidence and reassurance": [
        "CONFIDENCE_INCREASE",
        "CONFIDENCE_THROUGH_CONFIRMATION",
        "SOLUTION_CROSS_CHECKING",
        "REASSURANCE",
    ],

    "Competence threat and future anxiety": [
        "CONFIDENCE_DECREASE",
        "PERFORMANCE_COMPARISON_ANXIETY",
        "IMPOSTOR_FEELINGS",
        "AI_EXPECTATION_PRESSURE",
        "CAREER_FUTURE_ANXIETY",
    ],

    "Dependency and erosion of mastery": [
        "DEPENDENCY_OVERRELIANCE",
        "REDUCED_INDEPENDENT_THINKING",
        "LEARNING_REDUCTION",
        "UNDERSTANDING_GAP",
        "SKILL_PRACTICE_LOSS",
        "SKILL_LOSS_DESKILLING",
        "LOSS_OF_CODE_CONTROL",
        "DEPENDENCY_LOOP",
        "SHORT_TERM_RELIEF_LONG_TERM_COST",
    ],

    "Selective use and retained agency": [
        "INDEPENDENCE_MAINTAINED",
        "STRATEGIC_SELECTIVE_USE",
        "RESPONSIBLE_AI_INTEGRATION",
    ],

    "Trust, verification, and expertise": [
        "UNRELIABLE_INCORRECT_OUTPUT",
        "VERIFICATION_REQUIRED",
        "VERIFICATION_ANXIETY",
        "CRITICAL_THINKING_REQUIRED",
        "AI_LIMIT_AWARENESS",
        "NOVICE_VULNERABILITY",
        "EXPERTISE_ENABLES_VERIFICATION",
        "BLIND_ACCEPTANCE_RISK",
    ],
}

print("Provisional analytic clusters:", len(analytic_clusters))

In [ ]:
# Shared evidence lookup for candidate theme reviews.
# Codes and interpretations remain exactly as documented above.
def show_candidate_evidence(cluster_name, preview_rows=15):
    if cluster_name not in analytic_clusters:
        raise KeyError(f"Unknown candidate cluster: {cluster_name}")
    evidence = (
        coded_highlights.loc[
            coded_highlights["code"].isin(analytic_clusters[cluster_name]),
            ["highlight_id", "participant_id", "response_id",
             "document", "code", "highlight_clean"],
        ]
        .drop_duplicates()
        .sort_values(["highlight_id", "code"])
        .reset_index(drop=True)
    )
    print(f"{cluster_name}: {evidence['highlight_id'].nunique()} highlights, "
          f"{evidence['participant_id'].nunique()} participants")
    # Show excerpts locally for qualitative audit; clear notebook outputs before publication.
    if preview_rows:
        display(evidence.head(preview_rows))
    return evidence


In [ ]:
# Build excerpt banks for each analytic cluster
# The purpose of this step is to inspect whether the grouped
# codes actually express a coherent shared pattern when the
# participant excerpts are read together.

cluster_excerpt_banks = {}

for cluster_name, cluster_codes in analytic_clusters.items():

    subset = (
        coded_highlights[
            coded_highlights["code"].isin(cluster_codes)
        ]
        [
            [
                "participant_id",
                "response_id",
                "document",
                "code",
                "highlight_clean",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    cluster_excerpt_banks[cluster_name] = subset

    print("\n" + "=" * 100)
    print(cluster_name.upper())
    print("=" * 100)

    print(
        "Excerpts:",
        len(subset),
        "| Participants:",
        subset["participant_id"].nunique()
    )

    display(subset.head(20))

In [ ]:
# 12c. record candidate themes after excerpt review
# After reviewing the excerpt banks above, a candidate theme is
# recorded only where the excerpts form a coherent pattern of
# shared meaning.
#
# The theme label summarises the underlying meaning observed
# across the excerpts rather than merely repeating the code names.
#
# This remains a provisional stage of the thematic analysis.
# Candidate themes may still be merged, refined, or reorganised
# after comparing them with one another and reviewing their
# boundaries and distinctiveness.

candidate_theme_interpretations = pd.DataFrame(
    [
        {
            "cluster": "Support and manageability",
            "candidate_theme":
                "AI as a safety net that makes difficult tasks feel manageable",
            "shared_meaning":
                "AI is experienced as a fallback that increases perceived control, "
                "reduces the risk of remaining stuck, and makes difficult tasks feel "
                "more achievable."
        },

        {
            "cluster": "Learning access and explanation",
            "candidate_theme":
                "AI lowers barriers to learning through accessible and personalized support",
            "shared_meaning":
                "AI makes academic help easier to access and adapt to individual needs "
                "through repeated questioning, explanations, tutoring, and task support."
        },

        {
            "cluster": "Confidence and reassurance",
            "candidate_theme":
                "AI can scaffold confidence through confirmation and support",
            "shared_meaning":
                "Confidence is strengthened when AI validates or supports the participant's "
                "own reasoning rather than replacing it."
        },

        {
            "cluster": "Competence threat and future anxiety",
            "candidate_theme":
                "AI can become a benchmark that threatens perceived competence and future value",
            "shared_meaning":
                "AI becomes a comparison standard that can make participants question their "
                "own competence, performance, and future relevance."
        },

        {
            "cluster": "Dependency and erosion of mastery",
            "candidate_theme":
                "Convenience can erode independent mastery and create dependency",
            "shared_meaning":
                "The convenience of AI can reduce practice, understanding, and independent "
                "problem solving, creating greater reliance on AI over time."
        },

        {
            "cluster": "Selective use and retained agency",
            "candidate_theme":
                "Students preserve agency by using AI selectively rather than as a replacement",
            "shared_meaning":
                "Participants describe deliberate boundaries around AI use in order to retain "
                "understanding, ownership, and independent reasoning."
        },

        {
            "cluster": "Trust, verification, and expertise",
            "candidate_theme":
                "Trust in AI depends on the user's ability to verify it",
            "shared_meaning":
                "AI can only be used confidently when its output can be critically evaluated, "
                "and this ability depends partly on prior knowledge and expertise."
        },
    ]
)

# Start the displayed table index at 1 rather than 0.
candidate_theme_interpretations.index = (
    candidate_theme_interpretations.index + 1
)

print(
    "Candidate themes recorded after excerpt review:",
    len(candidate_theme_interpretations)
)

display(candidate_theme_interpretations)

In [ ]:
# Review evidence for each candidate theme
# This step evaluates each provisional theme against the actual
# coded excerpts that support it.
#
# For each candidate theme we summarise:
#   - number of supporting codes
#   - number of unique coded highlights
#   - number of unique participants
#   - number of qualitative documents represented
#
# These counts do not decide whether a theme is valid.
# They provide context for the manual review of coherence,
# distinctiveness, and relevance.

candidate_review_rows = []

for cluster_name, cluster_codes in analytic_clusters.items():

    evidence = coded_highlights[
        coded_highlights["code"].isin(cluster_codes)
    ].copy()

    candidate_theme = (
        candidate_theme_interpretations.loc[
            candidate_theme_interpretations["cluster"] == cluster_name,
            "candidate_theme"
        ]
        .iloc[0]
    )

    candidate_review_rows.append(
        {
            "cluster": cluster_name,
            "candidate_theme": candidate_theme,
            "n_supporting_codes": len(cluster_codes),
            "n_highlights": evidence["highlight_id"].nunique(),
            "n_participants": evidence["participant_id"].nunique(),
            "n_documents": evidence["document"].nunique(),
        }
    )

candidate_theme_review = pd.DataFrame(
    candidate_review_rows
)

candidate_theme_review.index = (
    candidate_theme_review.index + 1
)

display(candidate_theme_review)

In [ ]:
# Refine support-and-manageability cluster
# Review of the excerpts showed that STRESS_REDUCTION was too
# broad to function as a defining code for this candidate theme.
#
# Some participants reported reduced stress for reasons unrelated
# to AI providing support (for example, reducing their own AI use).
#
# The candidate theme is therefore defined primarily by the
# mechanisms through which AI makes tasks feel more manageable.
# Stress reduction remains an important associated outcome, but
# is not used by itself as evidence for this theme.

analytic_clusters[
    "Support and manageability"
] = [
    "TASK_COMPLETION_SAFETY_NET",
    "TIME_PRESSURE_RELIEF",
    "BREAKING_TASK_PARALYSIS",
    "ALWAYS_AVAILABLE_SUPPORT",
    "REASSURANCE",
    "TASK_CLARIFICATION",
]

print(
    "Refined supporting codes for "
    "'Support and manageability':"
)

for code_name in analytic_clusters[
    "Support and manageability"
]:
    print("-", code_name)

In [ ]:
support_manageability_evidence = show_candidate_evidence(
    "Support and manageability"
)

In [ ]:
# Record review decision for candidate 1
# The candidate theme was reviewed against its underlying
# participant excerpts.
#
# STRESS_REDUCTION was removed as a defining code because it was
# too broad and sometimes reflected reduced stress for reasons
# unrelated to AI providing support.
#
# The remaining codes formed a more coherent pattern centred on
# availability, reassurance, task clarification, time-pressure
# relief, and the ability to continue when stuck.

candidate_1_review = {
    "cluster": "Support and manageability",
    "candidate_theme":
        "AI as a safety net that makes difficult tasks feel manageable",
    "internal_coherence": "Strong",
    "distinctiveness": "Strong",
    "evidence_strength": "Strong",
    "decision": "Retain with refinement",
    "refinement":
        "Removed STRESS_REDUCTION as a defining code and retained it as a broader associated outcome.",
    "review_summary":
        "The remaining excerpts consistently describe AI as a fallback or support mechanism that increases perceived control, helps participants continue when stuck, clarifies difficult tasks, and reduces pressure around successful task completion."
}

display(
    pd.DataFrame(
        [candidate_1_review],
        index=[1]
    )
)

In [ ]:
# Inspect candidate 2: learning access and explanation
# This candidate groups excerpts describing AI as an accessible,
# adaptable learning resource.
#
# We review the underlying excerpts to determine whether they form
# a coherent pattern of shared meaning and whether the cluster is
# distinct from the support-and-manageability candidate.

learning_access_evidence = show_candidate_evidence(
    "Learning access and explanation"
)

In [ ]:
# Record review decision for candidate 2
# Review of the underlying excerpts showed a strong and coherent
# pattern centred on accessible, adaptable learning support.
#
# Participants repeatedly described AI as a resource for:
#   - understanding difficult concepts
#   - receiving personalized explanations
#   - asking repeated or highly specific questions
#   - working with tutor-like support
#   - understanding code and errors
#   - checking their own knowledge
#
# TASK_STRUCTURING is retained where it supports learning or
# understanding. Structuring used primarily to make task
# completion easier belongs more closely to the support-and-
# manageability candidate.

candidate_2_review = {
    "cluster": "Learning access and explanation",

    "candidate_theme":
        "AI lowers barriers to learning through accessible and personalized support",

    "internal_coherence": "Strong",

    "distinctiveness": "Strong",

    "evidence_strength": "Very strong",

    "decision": "Retain",

    "refinement":
        "TASK_STRUCTURING is interpreted within this candidate only when it supports "
        "learning or understanding; task-completion-oriented structuring is more closely "
        "related to the support-and-manageability candidate.",

    "review_summary":
        "The excerpts consistently describe AI as an accessible and adaptable learning "
        "resource. Participants used it for personalized explanations, repeated questioning, "
        "tutoring, understanding concepts, code and errors, and checking their own knowledge. "
        "The shared meaning is therefore broader than generic learning support: AI lowers "
        "barriers to obtaining and tailoring academic help."
}

display(
    pd.DataFrame(
        [candidate_2_review],
        index=[2]
    )
)

In [ ]:
confidence_reassurance_evidence = show_candidate_evidence(
    "Confidence and reassurance"
)

In [ ]:
# Refine confidence-and-reassurance cluster
# Review of the underlying excerpts showed that
# CONFIDENCE_INCREASE is too broad to function as a defining
# code for this candidate theme.
#
# Participants reported increased confidence for several
# different reasons, not all of which involved confirmation or
# cross-checking.
#
# REASSURANCE was also found to overlap substantially with the
# support-and-manageability candidate, where reassurance often
# reflected knowing that help or a fallback was available.
#
# This candidate is therefore narrowed to the more specific
# mechanism through which AI validates the participant's own
# reasoning, understanding, or solution.
#
# CONFIDENCE_INCREASE remains an important associated outcome,
# but is not sufficient by itself as evidence for this theme.

analytic_clusters[
    "Confidence and reassurance"
] = [
    "CONFIDENCE_THROUGH_CONFIRMATION",
    "SOLUTION_CROSS_CHECKING",
]

print(
    "Refined supporting codes for "
    "'Confidence and reassurance':"
)

for code_name in analytic_clusters[
    "Confidence and reassurance"
]:
    print("-", code_name)

In [ ]:
confidence_reassurance_evidence = show_candidate_evidence(
    "Confidence and reassurance"
)

In [ ]:
# Record review decision for candidate 3
# Review of the refined evidence showed a coherent but relatively
# specific pattern: participants sometimes used AI as a second
# opinion to confirm their own reasoning, code, or understanding.
#
# CONFIDENCE_THROUGH_CONFIRMATION provides the clearest evidence
# for this mechanism.
#
# SOLUTION_CROSS_CHECKING supports the candidate where checking
# or comparison contributes to reassurance or confidence, but the
# code also contains broader verification behaviours that do not
# necessarily concern confidence.
#
# The candidate is therefore retained as a narrow confidence-
# related pattern rather than interpreted as a broad standalone
# account of confidence increase.

candidate_3_review = {
    "cluster": "Confidence and reassurance",

    "candidate_theme":
        "AI can scaffold confidence through confirmation",

    "internal_coherence": "Strong",

    "distinctiveness": "Moderate to strong",

    "evidence_strength": "Moderate to strong",

    "decision": "Retain with refinement",

    "refinement":
        "CONFIDENCE_INCREASE was removed as a defining code because it described "
        "the outcome without explaining its mechanism. REASSURANCE was also removed "
        "because it overlapped strongly with the support-and-manageability candidate. "
        "SOLUTION_CROSS_CHECKING is treated as supporting evidence only where checking "
        "or comparison contributes to reassurance or confidence.",

    "review_summary":
        "The most coherent excerpts describe AI functioning as a validating second "
        "opinion. Confidence increases when AI confirms, resembles, or validates the "
        "participant's own answer, reasoning, code, or understanding. The pattern is "
        "specific enough that it may be better represented as a subtheme within a "
        "broader theme concerning how AI reshapes confidence."
}

display(
    pd.DataFrame(
        [candidate_3_review],
        index=[3]
    )
)

In [ ]:
competence_threat_evidence = show_candidate_evidence(
    "Competence threat and future anxiety"
)

In [ ]:
# Record review decision for candidate 4
# Review of the evidence showed a coherent broader pattern in
# which AI becomes a comparison point for evaluating one's own
# competence, performance, and future value.
#
# Two closely related forms of this pattern were visible:
#
#   1. present competence threat
#      - self-doubt
#      - perceived underperformance
#      - impostor feelings
#      - comparison with AI or AI-assisted others
#
#   2. future-value threat
#      - career anxiety
#      - concerns about employability
#      - fear that one's skills may lose value
#
# CONFIDENCE_DECREASE is treated primarily as an associated
# outcome rather than the defining mechanism of the candidate.

candidate_4_review = {
    "cluster": "Competence threat and future anxiety",

    "candidate_theme":
        "AI can become a benchmark that threatens perceived competence and future value",

    "internal_coherence": "Strong",

    "distinctiveness": "Strong",

    "evidence_strength": "Strong",

    "decision": "Retain with internal differentiation",

    "refinement":
        "CONFIDENCE_DECREASE is treated as a broad associated outcome rather than "
        "the defining mechanism. The candidate contains two related sub-patterns: "
        "present competence threat and future-value or career threat.",

    "review_summary":
        "Across the excerpts, participants frequently evaluated themselves relative "
        "to AI, AI-generated performance, or AI-assisted others. This comparison could "
        "produce self-doubt, perceived underperformance, impostor feelings, and pressure "
        "to keep up. For some participants, the same process extended into concerns "
        "about future employability and whether their own skills would remain valuable."
}

display(
    pd.DataFrame(
        [candidate_4_review],
        index=[4]
    )
)

In [ ]:
dependency_mastery_evidence = show_candidate_evidence(
    "Dependency and erosion of mastery"
)

In [ ]:
# Refine dependency-and-mastery cluster
# Review of the underlying excerpts showed a strong process
# linking convenient AI assistance with reduced independent
# practice, weaker learning, skill deterioration, and increasing
# reliance on AI.
#
# UNDERSTANDING_GAP was removed as a defining code because it was
# broader than this process. Some excerpts carrying that code
# reflected limited prior knowledge or difficulty understanding
# unfamiliar material rather than erosion caused by AI use.
#
# Where lack of understanding clearly results from relying on AI,
# those excerpts can still contribute contextually through the
# remaining codes, particularly LEARNING_REDUCTION,
# DEPENDENCY_LOOP, and SHORT_TERM_RELIEF_LONG_TERM_COST.

analytic_clusters[
    "Dependency and erosion of mastery"
] = [
    "DEPENDENCY_OVERRELIANCE",
    "REDUCED_INDEPENDENT_THINKING",
    "LEARNING_REDUCTION",
    "SKILL_PRACTICE_LOSS",
    "SKILL_LOSS_DESKILLING",
    "LOSS_OF_CODE_CONTROL",
    "DEPENDENCY_LOOP",
    "SHORT_TERM_RELIEF_LONG_TERM_COST",
]

print(
    "Refined supporting codes for "
    "'Dependency and erosion of mastery':"
)

for code_name in analytic_clusters[
    "Dependency and erosion of mastery"
]:
    print("-", code_name)

In [ ]:
dependency_mastery_evidence = show_candidate_evidence(
    "Dependency and erosion of mastery"
)

In [ ]:
# Record review decision for candidate 5
# After refinement, the excerpts form a coherent process in which
# convenient AI assistance can reduce opportunities for independent
# reasoning and practice, weaken learning or skill retention, and
# increase reliance on AI.
#
# Several responses suggest a reinforcing cycle:
#
#   convenience / shortcut
#          ↓
#   reduced independent practice
#          ↓
#   weaker mastery or skill retention
#          ↓
#   greater reliance on AI
#
# UNDERSTANDING_GAP was removed as a defining code because some
# excerpts reflected unfamiliarity or limited prior knowledge rather
# than erosion caused by AI use itself.

candidate_5_review = {
    "cluster": "Dependency and erosion of mastery",

    "candidate_theme":
        "Convenience can erode independent mastery and create dependency",

    "internal_coherence": "Strong",

    "distinctiveness": "Strong",

    "evidence_strength": "Very strong",

    "decision": "Retain with refinement",

    "refinement":
        "UNDERSTANDING_GAP was removed as a defining code because some excerpts "
        "reflected unfamiliarity or limited prior knowledge rather than erosion "
        "caused by AI use.",

    "review_summary":
        "The refined excerpts describe a recurring trade-off between immediate "
        "convenience and independent mastery. Participants reported reduced thinking "
        "and practice, weaker learning, skill deterioration, loss of control over "
        "their own work, and increasing reliance on AI. Several accounts suggest "
        "a reinforcing cycle in which reduced independent ability creates further "
        "need for AI assistance."
}

display(
    pd.DataFrame(
        [candidate_5_review],
        index=[5]
    )
)

In [ ]:
selective_agency_evidence = show_candidate_evidence(
    "Selective use and retained agency"
)

In [ ]:
# Record review decision for candidate 6
# Review of the underlying excerpts showed a coherent pattern of
# deliberate boundary-setting around AI use.
#
# Participants described:
#   - attempting problems independently before asking AI
#   - restricting AI to particular tasks or situations
#   - avoiding full replacement of their own reasoning
#   - reducing AI use when they felt it was becoming harmful
#   - using AI as assistance rather than as the primary agent
#
# The shared meaning is not simply "low AI use". It is the active
# preservation of autonomy, understanding, and ownership.

candidate_6_review = {
    "cluster": "Selective use and retained agency",

    "candidate_theme":
        "Students preserve agency by using AI selectively rather than as a replacement",

    "internal_coherence": "Strong",

    "distinctiveness": "Strong",

    "evidence_strength": "Strong",

    "decision": "Retain",

    "refinement":
        "No major code removal was necessary. STRATEGIC_SELECTIVE_USE is interpreted "
        "as deliberate boundary-setting rather than merely infrequent AI use.",

    "review_summary":
        "The excerpts consistently describe students exercising control over when and "
        "how AI is used. Participants attempted tasks independently first, restricted "
        "AI to selected functions, avoided complete replacement of their own reasoning, "
        "and sometimes reduced use when they perceived negative consequences. The "
        "shared meaning is the deliberate preservation of agency, understanding, and "
        "ownership while still benefiting from AI assistance."
}

display(
    pd.DataFrame(
        [candidate_6_review],
        index=[6]
    )
)

In [ ]:
trust_verification_evidence = show_candidate_evidence(
    "Trust, verification, and expertise"
)

In [ ]:
# Record review decision for candidate 7
# Review of the evidence showed a strong and coherent pattern
# centred on the conditional nature of trust in AI-generated
# output.
#
# Two closely connected sub-patterns were visible:
#
#   1. fallibility and verification
#      - incorrect or hallucinated output
#      - need for checking and cross-referencing
#      - critical thinking
#      - risks of blindly accepting AI output
#      - anxiety when correctness cannot be established
#
#   2. expertise and evaluative ability
#      - experienced users may be better able to identify errors
#      - limited prior knowledge can make verification difficult
#      - awareness of AI limitations influences how cautiously
#        participants use and trust its output
#
# These sub-patterns share the broader meaning that AI cannot be
# trusted independently of the user's ability to evaluate it.

candidate_7_review = {
    "cluster": "Trust, verification, and expertise",

    "candidate_theme":
        "Trust in AI depends on the user's ability to verify it",

    "internal_coherence": "Strong",

    "distinctiveness": "Strong",

    "evidence_strength": "Very strong",

    "decision": "Retain with internal differentiation",

    "refinement":
        "No major defining code was removed. The candidate is interpreted as "
        "containing two related sub-patterns: fallibility and verification, and "
        "the role of prior knowledge or expertise in evaluating AI output.",

    "review_summary":
        "Participants repeatedly described AI as fallible and therefore requiring "
        "checking, testing, cross-referencing, and critical evaluation. Blind "
        "acceptance was treated as risky, and uncertainty about correctness could "
        "itself create anxiety. The evidence also suggests that the ability to "
        "evaluate AI output is uneven: greater prior knowledge or experience can "
        "support confident verification, whereas limited knowledge can make errors "
        "more difficult to recognise."
}

display(
    pd.DataFrame(
        [candidate_7_review],
        index=[7]
    )
)

In [ ]:
# Combine all candidate theme review decisions
# Each of the seven candidate themes has now been reviewed
# individually against its supporting excerpts.
#
# The review considered:
#   - internal coherence
#   - distinctiveness from other candidate themes
#   - strength of the supporting evidence
#   - whether refinement or internal differentiation was needed
#
# This step combines the seven individual review records into one
# summary table.
#
# IMPORTANT:
# The candidate themes have not yet been reorganised into the
# final thematic structure. This table provides a transparent
# record of the review process before cross-candidate synthesis.

candidate_reviews = pd.DataFrame(
    [
        candidate_1_review,
        candidate_2_review,
        candidate_3_review,
        candidate_4_review,
        candidate_5_review,
        candidate_6_review,
        candidate_7_review,
    ],
    index=range(1, 8)
)

print(
    "Candidate themes reviewed:",
    len(candidate_reviews)
)

display(
    candidate_reviews[
        [
            "cluster",
            "candidate_theme",
            "internal_coherence",
            "distinctiveness",
            "evidence_strength",
            "decision",
            "refinement",
            "review_summary",
        ]
    ]
)

In [ ]:
# Synthesise candidate themes into broader final themes
# The seven candidate themes have now been reviewed individually.
#
# This step compares them with one another to determine whether
# conceptually related candidates should remain separate final
# themes or be organised as subthemes within a broader pattern.
#
# The synthesis is interpretive rather than statistical.
#
# Decisions are based on:
#   - the shared meaning identified in the excerpts
#   - conceptual overlap between candidate themes
#   - distinctions identified during candidate review
#   - whether candidates represent complementary or contrasting
#     parts of a broader process
#
# Four broader themes were identified:
#
#   Theme 1:
#       accessible support and manageability
#
#   Theme 2:
#       changes in confidence and perceived competence
#
#   Theme 3:
#       tension between AI assistance and independent mastery
#
#   Theme 4:
#       conditional trust, verification, and prior knowledge
#
# The table below documents exactly how each of the seven reviewed
# candidates contributes to this final structure.

theme_synthesis = pd.DataFrame(
    [
        {
            "candidate_id": 1,
            "candidate_cluster":
                "Support and manageability",

            "reviewed_candidate":
                "AI as a safety net that makes difficult tasks feel manageable",

            "synthesis_decision":
                "Retain as subtheme",

            "final_theme_id":
                "Theme 1",

            "final_theme":
                "AI as accessible support that makes difficult tasks feel manageable",

            "proposed_subtheme":
                "1b. AI as a safety net under uncertainty and time pressure",

            "synthesis_rationale":
                "This candidate shares with the learning-access candidate the broader "
                "idea that accessible AI support can reduce barriers to progressing "
                "through difficult academic tasks. Its distinctive focus is the "
                "experience of AI as a fallback when participants are stuck, uncertain, "
                "or under time pressure."
        },

        {
            "candidate_id": 2,
            "candidate_cluster":
                "Learning access and explanation",

            "reviewed_candidate":
                "AI lowers barriers to learning through accessible and personalized support",

            "synthesis_decision":
                "Retain as subtheme",

            "final_theme_id":
                "Theme 1",

            "final_theme":
                "AI as accessible support that makes difficult tasks feel manageable",

            "proposed_subtheme":
                "1a. Low-barrier, personalized learning support",

            "synthesis_rationale":
                "This candidate also concerns accessible support but is specifically "
                "centred on learning and understanding. Personalized explanations, "
                "repeated questioning, tutoring, code and error explanation, and "
                "knowledge checking distinguish it from the safety-net pattern while "
                "still supporting the same broader theme."
        },

        {
            "candidate_id": 3,
            "candidate_cluster":
                "Confidence and reassurance",

            "reviewed_candidate":
                "AI can scaffold confidence through confirmation",

            "synthesis_decision":
                "Retain as subtheme",

            "final_theme_id":
                "Theme 2",

            "final_theme":
                "AI reshapes confidence through reassurance and comparison",

            "proposed_subtheme":
                "2a. Confidence through confirmation",

            "synthesis_rationale":
                "This candidate represents a positive pathway through which AI can "
                "affect perceived competence. AI can function as a validating second "
                "opinion when it confirms participants' own reasoning, understanding, "
                "or solutions."
        },

        {
            "candidate_id": 4,
            "candidate_cluster":
                "Competence threat and future anxiety",

            "reviewed_candidate":
                "AI can become a benchmark that threatens perceived competence and future value",

            "synthesis_decision":
                "Retain as subtheme",

            "final_theme_id":
                "Theme 2",

            "final_theme":
                "AI reshapes confidence through reassurance and comparison",

            "proposed_subtheme":
                "2b. Threats to competence, expectations, and future value",

            "synthesis_rationale":
                "This candidate represents the contrasting negative pathway. "
                "Participants could evaluate themselves against AI, AI-generated "
                "performance, or AI-assisted others, producing self-doubt, perceived "
                "underperformance, heightened expectations, and concerns about future "
                "skill value or employability."
        },

        {
            "candidate_id": 5,
            "candidate_cluster":
                "Dependency and erosion of mastery",

            "reviewed_candidate":
                "Convenience can erode independent mastery and create dependency",

            "synthesis_decision":
                "Retain as subtheme",

            "final_theme_id":
                "Theme 3",

            "final_theme":
                "Convenience creates a tension between assistance and independent mastery",

            "proposed_subtheme":
                "3a. Dependency, reduced practice, and loss of control",

            "synthesis_rationale":
                "This candidate describes one side of the broader tension created by "
                "AI assistance. Convenience can reduce independent reasoning, practice, "
                "learning, and control, sometimes contributing to increasing dependence "
                "on AI."
        },

        {
            "candidate_id": 6,
            "candidate_cluster":
                "Selective use and retained agency",

            "reviewed_candidate":
                "Students preserve agency by using AI selectively rather than as a replacement",

            "synthesis_decision":
                "Retain as subtheme",

            "final_theme_id":
                "Theme 3",

            "final_theme":
                "Convenience creates a tension between assistance and independent mastery",

            "proposed_subtheme":
                "3b. Selective use as a strategy for preserving agency",

            "synthesis_rationale":
                "This candidate represents the counter-strategy to the dependency "
                "pattern. Participants described deliberate boundaries around AI use "
                "to preserve their own reasoning, understanding, practice, and ownership "
                "while still benefiting from assistance."
        },

        {
            "candidate_id": 7,
            "candidate_cluster":
                "Trust, verification, and expertise",

            "reviewed_candidate":
                "Trust in AI depends on the user's ability to verify it",

            "synthesis_decision":
                "Retain as independent final theme",

            "final_theme_id":
                "Theme 4",

            "final_theme":
                "Trust in AI requires verification and sufficient prior knowledge",

            "proposed_subtheme":
                "Split into two internal subthemes",

            "synthesis_rationale":
                "This candidate remained conceptually distinct from the other patterns. "
                "Its organising idea concerns the conditions under which AI-generated "
                "output can be trusted. Review identified two related dimensions: the "
                "need to verify fallible AI output and differences in users' ability "
                "to perform that verification depending on prior knowledge or expertise."
        },
    ]
)

theme_synthesis = theme_synthesis.set_index(
    "candidate_id"
)

display(
    theme_synthesis[
        [
            "candidate_cluster",
            "reviewed_candidate",
            "synthesis_decision",
            "final_theme_id",
            "final_theme",
            "proposed_subtheme",
            "synthesis_rationale",
        ]
    ]
)

In [ ]:
# Define the final thematic structure
# After reviewing the seven candidate themes and comparing their
# conceptual relationships, the analysis was refined into four
# final themes containing eight subthemes.
#
# This final structure reflects:
#   - the shared meaning across supporting excerpts
#   - distinctions between related patterns
#   - the synthesis decisions documented in Step 22
#
# The table below represents the final thematic interpretation
# that will be used in the qualitative Results section.

final_thematic_structure = pd.DataFrame(
    [
        {
            "theme_id": "Theme 1",
            "final_theme":
                "AI as accessible support that makes difficult tasks feel manageable",
            "subtheme":
                "1a. Low-barrier, personalized learning support",
            "central_idea":
                "AI provides accessible and adaptable academic support through "
                "personalized explanations, repeated questioning, tutor-like interaction, "
                "task structuring, code and error explanations, and knowledge checking."
        },

        {
            "theme_id": "Theme 1",
            "final_theme":
                "AI as accessible support that makes difficult tasks feel manageable",
            "subtheme":
                "1b. AI as a safety net under uncertainty and time pressure",
            "central_idea":
                "Knowing that AI is available as a fallback can increase perceived "
                "control, reduce fear of remaining stuck, relieve pressure around "
                "deadlines or difficult tasks, and make successful task completion "
                "feel more attainable."
        },

        {
            "theme_id": "Theme 2",
            "final_theme":
                "AI reshapes confidence through reassurance and comparison",
            "subtheme":
                "2a. Confidence through confirmation",
            "central_idea":
                "AI can strengthen confidence when it confirms, resembles, or validates "
                "the participant's own reasoning, understanding, code, or solution, "
                "allowing it to function as a validating second opinion."
        },

        {
            "theme_id": "Theme 2",
            "final_theme":
                "AI reshapes confidence through reassurance and comparison",
            "subtheme":
                "2b. Threats to competence, expectations, and future value",
            "central_idea":
                "Comparison with AI, AI-generated performance, or AI-assisted others "
                "can produce self-doubt, perceived underperformance, heightened "
                "expectations, impostor feelings, and concerns about future skill "
                "value or employability."
        },

        {
            "theme_id": "Theme 3",
            "final_theme":
                "Convenience creates a tension between assistance and independent mastery",
            "subtheme":
                "3a. Dependency, reduced practice, and loss of control",
            "central_idea":
                "Heavy or convenient reliance on AI can reduce independent reasoning "
                "and practice, weaken learning or skill retention, reduce ownership "
                "over the work, and contribute to increasing dependence on AI."
        },

        {
            "theme_id": "Theme 3",
            "final_theme":
                "Convenience creates a tension between assistance and independent mastery",
            "subtheme":
                "3b. Selective use as a strategy for preserving agency",
            "central_idea":
                "Participants described deliberate boundaries around AI use, including "
                "attempting work independently first, restricting AI to selected tasks, "
                "avoiding full replacement of their own reasoning, and ensuring that "
                "they retained understanding and ownership."
        },

        {
            "theme_id": "Theme 4",
            "final_theme":
                "Trust in AI requires verification and sufficient prior knowledge",
            "subtheme":
                "4a. Fallibility, checking, and verification anxiety",
            "central_idea":
                "Because AI can produce incorrect, misleading, or hallucinated output, "
                "participants described checking, testing, and cross-referencing as "
                "necessary. For some, uncertainty about whether AI was correct could "
                "itself create stress or anxiety."
        },

        {
            "theme_id": "Theme 4",
            "final_theme":
                "Trust in AI requires verification and sufficient prior knowledge",
            "subtheme":
                "4b. Novice vulnerability versus expertise-supported evaluation",
            "central_idea":
                "The ability to identify unreliable AI output depends partly on prior "
                "knowledge and experience. Limited knowledge can make errors harder to "
                "recognise, whereas greater expertise can support more confident and "
                "selective evaluation."
        },
    ]
)

# Start displayed row numbers at 1.
final_thematic_structure.index = (
    final_thematic_structure.index + 1
)

print(
    "Final themes:",
    final_thematic_structure["theme_id"].nunique()
)

print(
    "Final subthemes:",
    len(final_thematic_structure)
)

display(final_thematic_structure)

In [ ]:
# Map supporting codes to the final subthemes
# The final thematic structure has now been defined.
#
# This step maps the relevant qualitative codes onto each final
# subtheme so that the breadth of supporting evidence can be
# summarised systematically.
#
# These mappings are based on the candidate-theme review and
# refinements completed in the previous steps.
#
# IMPORTANT:
# The counts produced from these mappings describe the amount
# and distribution of supporting coded material. They do not
# determine the importance of a theme by themselves.

subtheme_code_map = {

    # --------------------------------------------------------
    # THEME 1
    # AI as accessible support that makes difficult tasks feel
    # manageable
    # --------------------------------------------------------

    "1a. Low-barrier, personalized learning support": [
        "LEARNING_SUPPORT",
        "PERSONALIZED_EXPLANATION",
        "AI_AS_TUTOR",
        "LOW_BARRIER_QUESTIONING",
        "TASK_STRUCTURING",
        "LEARNING_CATCH_UP",
        "CODE_EXPLANATION",
        "ERROR_EXPLANATION",
        "KNOWLEDGE_CHECKING",
    ],

    "1b. AI as a safety net under uncertainty and time pressure": [
        "TASK_COMPLETION_SAFETY_NET",
        "TIME_PRESSURE_RELIEF",
        "BREAKING_TASK_PARALYSIS",
        "ALWAYS_AVAILABLE_SUPPORT",
        "REASSURANCE",
        "TASK_CLARIFICATION",
    ],

    # --------------------------------------------------------
    # THEME 2
    # AI reshapes confidence through reassurance and comparison
    # --------------------------------------------------------

    "2a. Confidence through confirmation": [
        "CONFIDENCE_THROUGH_CONFIRMATION",
        "SOLUTION_CROSS_CHECKING",
    ],

    "2b. Threats to competence, expectations, and future value": [
        "PERFORMANCE_COMPARISON_ANXIETY",
        "IMPOSTOR_FEELINGS",
        "AI_EXPECTATION_PRESSURE",
        "CAREER_FUTURE_ANXIETY",
    ],

    # --------------------------------------------------------
    # THEME 3
    # Convenience creates a tension between assistance and
    # independent mastery
    # --------------------------------------------------------

    "3a. Dependency, reduced practice, and loss of control": [
        "DEPENDENCY_OVERRELIANCE",
        "REDUCED_INDEPENDENT_THINKING",
        "LEARNING_REDUCTION",
        "SKILL_PRACTICE_LOSS",
        "SKILL_LOSS_DESKILLING",
        "LOSS_OF_CODE_CONTROL",
        "DEPENDENCY_LOOP",
        "SHORT_TERM_RELIEF_LONG_TERM_COST",
    ],

    "3b. Selective use as a strategy for preserving agency": [
        "INDEPENDENCE_MAINTAINED",
        "STRATEGIC_SELECTIVE_USE",
        "RESPONSIBLE_AI_INTEGRATION",
    ],

    # --------------------------------------------------------
    # THEME 4
    # Trust in AI requires verification and sufficient prior
    # knowledge
    # --------------------------------------------------------

    "4a. Fallibility, checking, and verification anxiety": [
        "UNRELIABLE_INCORRECT_OUTPUT",
        "VERIFICATION_REQUIRED",
        "VERIFICATION_ANXIETY",
        "CRITICAL_THINKING_REQUIRED",
        "BLIND_ACCEPTANCE_RISK",
    ],

    "4b. Novice vulnerability versus expertise-supported evaluation": [
        "AI_LIMIT_AWARENESS",
        "NOVICE_VULNERABILITY",
        "EXPERTISE_ENABLES_VERIFICATION",
    ],
}

print(
    "Final subthemes mapped:",
    len(subtheme_code_map)
)

for subtheme, codes in subtheme_code_map.items():

    print("\n" + subtheme)

    for code in codes:
        print("  -", code)

In [ ]:
# Summarise supporting evidence for each final subtheme
# For each final subtheme, this step calculates:
#
#   - number of supporting codes
#   - number of unique coded excerpts
#   - number of unique participants
#   - number of qualitative documents represented
#
# Participant counts are particularly useful because multiple
# excerpts can come from the same person. Therefore,
# n_participants provides a clearer indication of how broadly
# a pattern occurred across the sample.
#
# These figures are descriptive and should not be interpreted
# as statistical prevalence estimates.

subtheme_evidence_rows = []

for subtheme, codes in subtheme_code_map.items():

    evidence = coded_highlights[
        coded_highlights["code"].isin(codes)
    ].copy()

    theme_row = (
        final_thematic_structure[
            final_thematic_structure["subtheme"] == subtheme
        ]
        .iloc[0]
    )

    subtheme_evidence_rows.append(
        {
            "theme_id":
                theme_row["theme_id"],

            "final_theme":
                theme_row["final_theme"],

            "subtheme":
                subtheme,

            "n_supporting_codes":
                len(codes),

            "n_highlights":
                evidence["highlight_id"].nunique(),

            "n_participants":
                evidence["participant_id"].nunique(),

            "n_documents":
                evidence["document"].nunique(),
        }
    )


subtheme_evidence_summary = pd.DataFrame(
    subtheme_evidence_rows
)

subtheme_evidence_summary.index = (
    subtheme_evidence_summary.index + 1
)

display(subtheme_evidence_summary)

In [ ]:
# Build excerpt banks for each final subtheme
# The final thematic structure and code mappings are now fixed.
#
# This step creates a reusable helper for retrieving the coded
# excerpts associated with any final subtheme.
#
# The purpose is to support:
#   - final theme validation
#   - representative quotation selection
#   - Results-section writing
#
# Quotations should be selected for clarity and representativeness,
# not simply because they are dramatic or memorable.

def show_subtheme_evidence(subtheme_name):

    codes = subtheme_code_map[subtheme_name]

    evidence = (
        coded_highlights[
            coded_highlights["code"].isin(codes)
        ]
        [
            [
                "participant_id",
                "response_id",
                "document",
                "code",
                "highlight_clean",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "participant_id",
                "response_id",
                "code",
            ]
        )
        .reset_index(drop=True)
    )

    print("=" * 100)
    print(subtheme_name.upper())
    print("=" * 100)

    print(
        "Supporting excerpts:",
        len(evidence)
    )

    print(
        "Unique participants:",
        evidence["participant_id"].nunique()
    )

    print(
        "Documents represented:",
        evidence["document"].nunique()
    )

    display(evidence)

    return evidence

In [ ]:
theme_1a_evidence = show_subtheme_evidence(
    "1a. Low-barrier, personalized learning support"
)

In [ ]:
# Create a quotation bank for subtheme 1a
# The evidence table above contains one row per relevant
# code assignment. This means that the same excerpt can appear
# more than once when it carries multiple codes.
#
# For quotation selection, we want to inspect each distinct
# excerpt only once.
#
# This step therefore:
#   - keeps the participant and response identifiers
#   - keeps the original qualitative document
#   - collapses duplicate excerpt text
#   - records all relevant codes attached to that excerpt
#
# This produces a cleaner quotation bank for manual review.
#
# IMPORTANT:
# Quotes are selected based on how clearly and representatively
# they express the subtheme, not simply on frequency or wording.

subtheme_1a_codes = subtheme_code_map[
    "1a. Low-barrier, personalized learning support"
]

subtheme_1a_raw = coded_highlights[
    coded_highlights["code"].isin(subtheme_1a_codes)
].copy()


subtheme_1a_quote_bank = (
    subtheme_1a_raw
    .groupby(
        [
            "participant_id",
            "response_id",
            "document",
            "highlight_clean",
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        supporting_codes=(
            "code",
            lambda x: "; ".join(
                sorted(set(x))
            )
        )
    )
)


# Add a simple text-length measure.
# This is NOT used to determine thematic importance.
# It is only useful when looking for quotations that contain
# enough context to be understandable in the Results section.

subtheme_1a_quote_bank["word_count"] = (
    subtheme_1a_quote_bank["highlight_clean"]
    .str.split()
    .str.len()
)


subtheme_1a_quote_bank = (
    subtheme_1a_quote_bank
    .sort_values(
        [
            "participant_id",
            "response_id",
        ]
    )
    .reset_index(drop=True)
)

# Start displayed row numbering at 1.
subtheme_1a_quote_bank.index = (
    subtheme_1a_quote_bank.index + 1
)


print(
    "Distinct quotation candidates:",
    len(subtheme_1a_quote_bank)
)

print(
    "Participants represented:",
    subtheme_1a_quote_bank["participant_id"].nunique()
)

display(
    subtheme_1a_quote_bank[
        [
            "participant_id",
            "response_id",
            "document",
            "supporting_codes",
            "word_count",
            "highlight_clean",
        ]
    ]
)

In [ ]:
# Create a review shortlist for subtheme 1a
# Very short coded fragments can be analytically meaningful but
# may be difficult to understand when quoted independently in the
# thesis.
#
# For quotation review only, this step displays excerpts containing
# at least eight words.
#
# No data are deleted. The complete quotation bank from Step 27
# remains unchanged.

subtheme_1a_review_shortlist = (
    subtheme_1a_quote_bank[
        subtheme_1a_quote_bank["word_count"] >= 8
    ]
    .copy()
)

print(
    "Quotation candidates with at least 8 words:",
    len(subtheme_1a_review_shortlist)
)

display(
    subtheme_1a_review_shortlist[
        [
            "participant_id",
            "response_id",
            "document",
            "supporting_codes",
            "highlight_clean",
        ]
    ]
)

In [ ]:
# Create a balanced quotation review set for subtheme 1a
# Subtheme 1a contains several related forms of learning support.
# Rather than selecting quotations from the full pool of 111
# longer excerpts, this step creates a smaller review set that
# represents the different analytic facets of the subtheme.
#
# The aim is to avoid choosing quotations only because one type
# of response is especially frequent or memorable.
#
# The facets reviewed here are:
#
#   - general learning support
#   - personalized explanation
#   - tutor-like interaction
#   - low-barrier questioning
#   - structuring learning material
#   - catching up / unfamiliar knowledge
#   - code explanation
#   - error explanation
#   - knowledge checking
#
# Up to five sufficiently contextual excerpts are displayed for
# each code.
#
# IMPORTANT:
# This is a REVIEW aid only.
# Python is not deciding which quotations are analytically "best".

subtheme_1a_review_sets = []

for code in subtheme_1a_codes:

    code_quotes = (
        subtheme_1a_quote_bank[
            subtheme_1a_quote_bank["supporting_codes"].str.contains(
                code,
                regex=False,
                na=False
            )
        ]
        .copy()
    )

    # Prefer excerpts that contain enough context to make sense
    # when read outside the original response.
    contextual_quotes = code_quotes[
        code_quotes["word_count"] >= 8
    ].copy()

    # If fewer than five contextual excerpts exist, retain all
    # available contextual excerpts.
    #
    # Sorting by word count is only a practical reading aid:
    # moderately detailed excerpts often provide more context.
    contextual_quotes = (
        contextual_quotes
        .sort_values(
            "word_count",
            ascending=False
        )
        .head(5)
    )

    contextual_quotes["review_facet"] = code

    subtheme_1a_review_sets.append(
        contextual_quotes
    )


subtheme_1a_balanced_review = pd.concat(
    subtheme_1a_review_sets,
    ignore_index=True
)

# Remove cases where the same participant excerpt was selected
# under more than one analytic facet.
subtheme_1a_balanced_review = (
    subtheme_1a_balanced_review
    .drop_duplicates(
        subset=[
            "participant_id",
            "response_id",
            "highlight_clean",
        ]
    )
    .reset_index(drop=True)
)

subtheme_1a_balanced_review.index = (
    subtheme_1a_balanced_review.index + 1
)

print(
    "Balanced quotation candidates:",
    len(subtheme_1a_balanced_review)
)

display(
    subtheme_1a_balanced_review[
        [
            "review_facet",
            "participant_id",
            "response_id",
            "document",
            "supporting_codes",
            "word_count",
            "highlight_clean",
        ]
    ]
)

### Restricted quotation selection and attribution

The researcher's final selection of representative quotations, response-level attribution checks, and source-text validation were performed in the **private** analysis notebook. Those cells contain verbatim participant responses paired with pseudonymous identifiers and are intentionally excluded from this public code release.

The public workflow below preserves the aggregate thematic summary and aggregate exports. It does **not** reproduce the private quotation shortlist, private attribution audit, or original source-text verification. A restricted dataset and research permissions are required to reproduce these steps.

In [ ]:
# Export aggregate thematic summaries without participant quotations.
# For full quotation attribution and verification, use the restricted private workflow.

QUALITATIVE_EXPORT_DIR = PROJECT_ROOT / "data" / "processed" / "qualitative" / "exports"
QUALITATIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

final_theme_summary = final_thematic_structure.merge(
    subtheme_evidence_summary[
        ["subtheme", "n_supporting_codes", "n_highlights", "n_participants", "n_documents"]
    ],
    on="subtheme",
    how="left",
)

aggregate_exports = {
    "01_candidate_theme_reviews.csv": candidate_reviews,
    "02_candidate_to_final_theme_synthesis.csv": theme_synthesis,
    "03_final_thematic_structure.csv": final_thematic_structure,
    "04_subtheme_evidence_summary.csv": subtheme_evidence_summary,
    "06_final_theme_summary.csv": final_theme_summary,
}
for filename, data in aggregate_exports.items():
    data.to_csv(QUALITATIVE_EXPORT_DIR / filename, index=True)

assert len(final_theme_summary) == 8
assert final_theme_summary["theme_id"].nunique() == 4
print("PASS: aggregate thematic summaries exported; restricted quotation exports omitted.")